In [ ]:
import os # belt-and-suspenders: clear any insecure Py4J hints 
os.environ.pop("PYSPARK_GATEWAY_PORT", None) 
os.environ.pop("PYSPARK_GATEWAY_SECRET", None) 

try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
      .master("spark://spark-master:7077")
      .appName("s3a-committers-stable")
      .config("spark.plugins", "org.apache.gluten.GlutenPlugin") 
      .config("spark.memory.offHeap.size", "8g")  
      .config("spark.shuffle.manager","org.apache.spark.shuffle.sort.ColumnarShuffleManager")  
      .config("spark.memory.offHeap.enabled", "true") 
      .config("spark.sql.ansi.enabled", "false") 
      .config("spark.driver.extraClassPath","/opt/spark/jars/gluten-velox-bundle-spark4.0_2.13-linux_amd64-1.6.0.jar")
      .config("spark.executor.extraClassPath","/opt/spark/jars/gluten-velox-bundle-spark4.0_2.13-linux_amd64-1.6.0.jar")
      .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
      .config("spark.hadoop.fs.s3a.bucket.all.committer.magic.enabled", "true")
      .config("spark.eventLog.enabled", "false")
      .config("spark.dynamicAllocation.enabled", "false")
      .config("spark.log.level", "warn")

      .getOrCreate()
)


Check Hadoop configs.

In [ ]:
hc = spark._jsc.hadoopConfiguration()
print("factory =", hc.get("mapreduce.outputcommitter.factory.scheme.s3a"))
print("algo    =", hc.get("mapreduce.fileoutputcommitter.algorithm.version"))
print("commit  =", hc.get("fs.s3a.committer.name"))
print("protocol=", spark.conf.get("spark.sql.sources.commitProtocolClass"))
print("parquet =", spark.conf.get("spark.sql.parquet.output.committer.class"))


**S3 bucket** where we are going to read and write. Then modify the role of the instance/container to be able to read and write on that bucket.

In [ ]:
s3_base = "<<YOUR-S3-BUCKET>>"

Let's run the test with **Velox Enabled**.

In [ ]:
from pyspark.sql import functions as F, types as T
import time

# --------- Tunables ----------
N_ROWS = 50_000_000   # bump this up/down depending on cluster
N_PROD = 50_000
N_CTRY = 32
N_CH   = 6
N_DAYS = 90
REPART = 256          # target partitions for wide stages
BUCKETS_FOR_PCTS = 2048
tag = str(int(time.time()))
out_base = f"{s3_base}/{tag}"
print("Writing results under:", out_base)

# --------- Synthetic, skewed FACT ----------
# We avoid UDFs; everything is Catalyst-friendly.
# Skew: make a Zipf-ish product distribution and slightly skewed countries/channels.
fact = (
    spark.range(N_ROWS)
      .select(
          (F.floor(F.pow(F.rand()*0.999999 + 1e-9, 3) * N_PROD)).alias("product_id"),  # skew to low IDs
          (F.floor(F.pow(F.rand()*0.999999 + 1e-9, 1.7) * N_CTRY)).alias("country_id"),
          (F.floor(F.pow(F.rand()*0.999999 + 1e-9, 1.4) * N_CH)).alias("channel_id"),
          (F.floor(F.rand() * N_DAYS)).alias("day_idx"),
          # economics
          (F.round(F.rand()*90 + 10, 2)).alias("price"),
          (F.floor(F.rand()*5 + 1)).alias("qty"),
          (F.round(F.rand()*0.25, 4)).alias("discount")
      )
      .withColumn("revenue", F.col("price") * F.col("qty") * (1 - F.col("discount")))
      .repartition(REPART)
      .persist()
)
print("Fact rows:", fact.count())

# --------- Tiny Dims ----------
countries = (spark.range(N_CTRY)
    .select(F.col("id").alias("country_id"),
            F.concat(F.lit("C"), F.lpad(F.col("id").cast("string"), 2, "0")).alias("country_name"),
            (F.rand()*0.25 + 0.75).alias("vat")))  # per-country VAT factor

channels = (spark.range(N_CH)
    .select(F.col("id").alias("channel_id"),
            F.array(F.lit("web"), F.lit("store"), F.lit("partner"), F.lit("marketplace"),
                    F.lit("app"), F.lit("phone")).getItem(F.col("id")).alias("channel_name")))

calendar = (
    spark.range(N_DAYS)
         .select(
             F.col("id").cast("int").alias("day_idx"),
             F.date_add(F.current_date(), F.lit(-N_DAYS) + F.col("id").cast("int")).alias("date_key")
         )
)

# Materialize small dims (broadcast candidates)
countries.cache().count()
channels.cache().count()
calendar.cache().count()

# Utility timer
def time_it(label, df):
    start = time.time()
    # trigger with a write to s3 and count to ensure full evaluation
    path = f"{out_base}/{label}"
    df.write.mode("overwrite").parquet(path)
    rows = spark.read.parquet(path).count()
    secs = time.time() - start
    print(f"{label}: {rows:,} rows -> {secs:.2f}s")
    return secs

# ------------------ Query A: heavy multi-agg groupBy ------------------
# Many aggregates that Gluten accelerates well: hash agg, projections, filters; plus approx functions.
qA = (
    fact
      .withColumn("is_discounted", (F.col("discount") > 0.05).cast("int"))
      .groupBy("country_id", "channel_id")
      .agg(
          F.count("*").alias("rows"),
          F.sum("qty").alias("sum_qty"),
          F.sum("revenue").alias("sum_revenue"),
          F.avg("price").alias("avg_price"),
          F.stddev_pop("price").alias("std_price"),
          F.approx_count_distinct("product_id").alias("card_products"),
          F.sum(F.when(F.col("discount") > 0, 1).otherwise(0)).alias("num_disc"),
          F.expr(f"percentile_approx(price, array(0.25,0.5,0.75), {BUCKETS_FOR_PCTS})").alias("price_quartiles")
      )
      .orderBy(F.desc("sum_revenue"))
)

tA = time_it("Q_A_groupby_multiagg", qA)

# ------------------ Query B: rollup (cubing-ish) ------------------
# Roll up day -> country -> product; compute revenue stats.
qB = (
    fact.join(calendar, "day_idx", "inner")
        .rollup("date_key", "country_id", "product_id")
        .agg(
            F.sum("revenue").alias("rev"),
            F.sum("qty").alias("qty"),
            F.avg("discount").alias("avg_disc")
        )
        .filter(F.col("rev").isNotNull())
        .orderBy(F.desc("rev"))
)

tB = time_it("Q_B_rollup", qB)

# ------------------ Query C: join + hash agg + sort top-K ------------------
# Star join fact->countries/channels->calendar, compute gross/net, then top categories.
qC = (
    fact.join(countries.hint("broadcast"), "country_id", "inner")
        .join(channels.hint("broadcast"), "channel_id", "inner")
        .join(calendar, "day_idx", "inner")
        .withColumn("gross_rev", F.col("revenue"))
        .withColumn("net_rev", F.col("revenue") / F.col("vat"))
        .groupBy("country_name", "channel_name", "date_key")
        .agg(
            F.sum("gross_rev").alias("gross"),
            F.sum("net_rev").alias("net"),
            F.countDistinct("product_id").alias("sku_seen")
        )
        .orderBy(F.desc("gross"))
        .limit(5000)  # force a wide sort/top-K
)

tC = time_it("Q_C_star_join_topk", qC)

print(f"\nWall times (sec): Q_A={tA:.2f}, Q_B={tB:.2f}, Q_C={tC:.2f}")
print("Outputs:", out_base)


Let's disable Velox. Then, re-run the test.

In [ ]:
spark.conf.set("spark.gluten.enabled", "false")

In [ ]:
from pyspark.sql import functions as F, types as T
import time

# --------- Tunables ----------
N_ROWS = 50_000_000   # bump this up/down depending on cluster
N_PROD = 50_000
N_CTRY = 32
N_CH   = 6
N_DAYS = 90
REPART = 256          # target partitions for wide stages
BUCKETS_FOR_PCTS = 2048
s3_base = "s3a://gluten-friends/demoas_bench2"
tag = str(int(time.time()))
out_base = f"{s3_base}/{tag}"
print("Writing results under:", out_base)

# --------- Synthetic, skewed FACT ----------
# We avoid UDFs; everything is Catalyst-friendly.
# Skew: make a Zipf-ish product distribution and slightly skewed countries/channels.
fact = (
    spark.range(N_ROWS)
      .select(
          (F.floor(F.pow(F.rand()*0.999999 + 1e-9, 3) * N_PROD)).alias("product_id"),  # skew to low IDs
          (F.floor(F.pow(F.rand()*0.999999 + 1e-9, 1.7) * N_CTRY)).alias("country_id"),
          (F.floor(F.pow(F.rand()*0.999999 + 1e-9, 1.4) * N_CH)).alias("channel_id"),
          (F.floor(F.rand() * N_DAYS)).alias("day_idx"),
          # economics
          (F.round(F.rand()*90 + 10, 2)).alias("price"),
          (F.floor(F.rand()*5 + 1)).alias("qty"),
          (F.round(F.rand()*0.25, 4)).alias("discount")
      )
      .withColumn("revenue", F.col("price") * F.col("qty") * (1 - F.col("discount")))
      .repartition(REPART)
      .persist()
)
print("Fact rows:", fact.count())

# --------- Tiny Dims ----------
countries = (spark.range(N_CTRY)
    .select(F.col("id").alias("country_id"),
            F.concat(F.lit("C"), F.lpad(F.col("id").cast("string"), 2, "0")).alias("country_name"),
            (F.rand()*0.25 + 0.75).alias("vat")))  # per-country VAT factor

channels = (spark.range(N_CH)
    .select(F.col("id").alias("channel_id"),
            F.array(F.lit("web"), F.lit("store"), F.lit("partner"), F.lit("marketplace"),
                    F.lit("app"), F.lit("phone")).getItem(F.col("id")).alias("channel_name")))

calendar = (
    spark.range(N_DAYS)
         .select(
             F.col("id").cast("int").alias("day_idx"),
             F.date_add(F.current_date(), F.lit(-N_DAYS) + F.col("id").cast("int")).alias("date_key")
         )
)

# Materialize small dims (broadcast candidates)
countries.cache().count()
channels.cache().count()
calendar.cache().count()

# Utility timer
def time_it(label, df):
    start = time.time()
    # trigger with a write to s3 and count to ensure full evaluation
    path = f"{out_base}/{label}"
    df.write.mode("overwrite").parquet(path)
    rows = spark.read.parquet(path).count()
    secs = time.time() - start
    print(f"{label}: {rows:,} rows -> {secs:.2f}s")
    return secs

# ------------------ Query A: heavy multi-agg groupBy ------------------
# Many aggregates that Gluten accelerates well: hash agg, projections, filters; plus approx functions.
qA = (
    fact
      .withColumn("is_discounted", (F.col("discount") > 0.05).cast("int"))
      .groupBy("country_id", "channel_id")
      .agg(
          F.count("*").alias("rows"),
          F.sum("qty").alias("sum_qty"),
          F.sum("revenue").alias("sum_revenue"),
          F.avg("price").alias("avg_price"),
          F.stddev_pop("price").alias("std_price"),
          F.approx_count_distinct("product_id").alias("card_products"),
          F.sum(F.when(F.col("discount") > 0, 1).otherwise(0)).alias("num_disc"),
          F.expr(f"percentile_approx(price, array(0.25,0.5,0.75), {BUCKETS_FOR_PCTS})").alias("price_quartiles")
      )
      .orderBy(F.desc("sum_revenue"))
)

tA = time_it("Q_A_groupby_multiagg", qA)

# ------------------ Query B: rollup (cubing-ish) ------------------
# Roll up day -> country -> product; compute revenue stats.
qB = (
    fact.join(calendar, "day_idx", "inner")
        .rollup("date_key", "country_id", "product_id")
        .agg(
            F.sum("revenue").alias("rev"),
            F.sum("qty").alias("qty"),
            F.avg("discount").alias("avg_disc")
        )
        .filter(F.col("rev").isNotNull())
        .orderBy(F.desc("rev"))
)

tB = time_it("Q_B_rollup", qB)

# ------------------ Query C: join + hash agg + sort top-K ------------------
# Star join fact->countries/channels->calendar, compute gross/net, then top categories.
qC = (
    fact.join(countries.hint("broadcast"), "country_id", "inner")
        .join(channels.hint("broadcast"), "channel_id", "inner")
        .join(calendar, "day_idx", "inner")
        .withColumn("gross_rev", F.col("revenue"))
        .withColumn("net_rev", F.col("revenue") / F.col("vat"))
        .groupBy("country_name", "channel_name", "date_key")
        .agg(
            F.sum("gross_rev").alias("gross"),
            F.sum("net_rev").alias("net"),
            F.countDistinct("product_id").alias("sku_seen")
        )
        .orderBy(F.desc("gross"))
        .limit(5000)  # force a wide sort/top-K
)

tC = time_it("Q_C_star_join_topk", qC)

print(f"\nWall times (sec): Q_A={tA:.2f}, Q_B={tB:.2f}, Q_C={tC:.2f}")
print("Outputs:", out_base)
